In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, mne
from mne.time_frequency import psd_array_welch
from src.load import list_subjects, load_subject
from src.channels import CHANNELS, SFREQ

N = 4 * SFREQ                         # samples per 4 s segment = 512
segments, subject, label = [], [], []
for _, k in pd.DataFrame(list_subjects()).iterrows():
    x = mne.filter.filter_data(load_subject(k.path), SFREQ, 1.0, 40.0, verbose=False)
    x = x[:, :60 * SFREQ]                               # keep the first 60 s only
    segments.append(x.reshape(19, 15, N).transpose(1, 0, 2))   # -> (15 segments, 19 ch, 512)
    subject += [k.subject_id] * 15
    label += [k.label] * 15

W = np.concatenate(segments)          # all segments: (1815, 19, 512)
subject, label = np.array(subject), np.array(label)
print(W.shape)

(1815, 19, 512)


In [2]:
psd, freqs = psd_array_welch(W, SFREQ, fmin=1, fmax=40, n_fft=N, verbose=False)
print(psd.shape)                      # (1815 segments, 19 channels, n frequencies)

def band(lo, hi):
    """Power between lo and hi Hz, for every segment and channel -> (1815, 19)."""
    return psd[..., (freqs >= lo) & (freqs < hi)].sum(axis=-1)

(1815, 19, 157)


In [3]:
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold

theta, alpha, beta = band(4, 8), band(8, 13), band(13, 30)
total_1_40 = band(1, 41)
A = np.hstack([theta/total_1_40, alpha/total_1_40, beta/total_1_40, theta/beta])
names = [f"{b}_{ch}" for b in ["theta", "alpha", "beta", "tbr"] for ch in CHANNELS]
print(A.shape, len(names))      # (1815, 76) 76

(1815, 76) 76


In [4]:
def test_shap(cv):
    """Train on 4 folds, explain the held-out fold. Repeat for all 5 folds."""
    S = np.zeros(A.shape)
    for train, test in cv.split(A, label, groups=subject):
        model = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42)
        model.fit(A[train], label[train])
        sv = shap.TreeExplainer(model).shap_values(A[test])
        if isinstance(sv, list): sv = sv[1]          # older shap: [control, ADHD]
        elif sv.ndim == 3:       sv = sv[..., 1]     # newer shap: (segments, features, 2)
        S[test] = sv                                 # keep the "towards ADHD" part
    return S

S_subwise = test_shap(StratifiedGroupKFold(5, shuffle=True, random_state=0))   # subject-wise split
S_segwise = test_shap(StratifiedKFold(5, shuffle=True, random_state=0))        # segments shuffled
print("done")

C:\Users\Rajesh Sampathkumar\miniconda3\envs\medimg\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


done


In [5]:
imp = pd.DataFrame({"subwise": np.abs(S_subwise).mean(axis=0),
                    "segwise": np.abs(S_segwise).mean(axis=0)}, index=names)
imp = imp / imp.sum()                     # each feature's share of the total credit

print("subject-wise split top 10\n", imp.sort_values("subwise", ascending=False).head(10).round(3))
print("\nsegment-wise split (leaky) top 10\n", imp.sort_values("segwise", ascending=False).head(10).round(3))

for col in ["subwise", "segwise"]:
    n50 = (imp[col].sort_values(ascending=False).cumsum() < 0.5).sum() + 1
    print(f"{col}: {n50} features hold half of all the credit")

subject-wise split top 10
            subwise  segwise
beta_Pz      0.051    0.057
tbr_P7       0.045    0.046
alpha_O1     0.043    0.044
beta_T8      0.036    0.038
beta_P8      0.033    0.034
alpha_Fp2    0.032    0.030
tbr_T7       0.031    0.030
tbr_Pz       0.031    0.036
beta_T7      0.026    0.027
beta_P7      0.023    0.023

segment-wise split (leaky) top 10
            subwise  segwise
beta_Pz      0.051    0.057
tbr_P7       0.045    0.046
alpha_O1     0.043    0.044
beta_T8      0.036    0.038
tbr_Pz       0.031    0.036
beta_P8      0.033    0.034
alpha_Fp2    0.032    0.030
tbr_T7       0.031    0.030
beta_T7      0.026    0.027
beta_P3      0.020    0.026
subwise: 19 features hold half of all the credit
segwise: 18 features hold half of all the credit
